# Fable → Qwen3-8B — pipeline completo no Colab L4

Execute as células de cima para baixo. O notebook reúne ambiente, download, auditoria, preprocessing, testes, treino, avaliação e exportação. As etapas caras/opcionais são controladas pelas flags da próxima célula.

In [ ]:
# Controles da execução
SMOKE_TEST = True            # 750 exemplos e 100 steps; use False no treino principal
EXCLUDE_GLINT = False        # True remove completamente Glint-Research/Fable-5-traces
RUN_MODEL_SMOKE = False      # carrega o Qwen3-8B em 4-bit e executa um forward curto
RUN_STAGE1 = True            # SFT principal em 4096 tokens
RUN_MODEL_COMPARISON = False # compara base e adapters; requer tempo/VRAM adicionais
RUN_STAGE2 = False           # continuação opcional em 8192 tokens
RUN_EXPORT = False           # exporta o melhor adapter para o Google Drive
RUN_AGENT_EVAL = False       # executa tarefas próprias do agent harness, quando disponíveis

print({
    'smoke_test': SMOKE_TEST,
    'stage1': RUN_STAGE1,
    'stage2': RUN_STAGE2,
    'comparison': RUN_MODEL_COMPARISON,
    'export': RUN_EXPORT,
})

## 1. Repositório, Drive e dependências

In [ ]:
from pathlib import Path
import base64, importlib.metadata, json, os, platform, signal, subprocess, sys
from datetime import datetime, timezone

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/devlucascfarias/logos-3.git'
PROJECT_ROOT = Path(os.environ.get('FABLE_PROJECT_ROOT', '/content/logos-3' if IN_COLAB else Path.cwd())).resolve()

if IN_COLAB:
    from google.colab import drive, userdata
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    else:
        print('Google Drive já está montado em /content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/fable-qwen-distillation')
else:
    DRIVE_ROOT = PROJECT_ROOT

# O repositório é privado. Leia GH_TOKEN/GITHUB_TOKEN dos Secrets sem colocá-lo na URL.
GITHUB_TOKEN = os.environ.get('GH_TOKEN') or os.environ.get('GITHUB_TOKEN')
if not GITHUB_TOKEN and IN_COLAB:
    for secret_name in ('GH_TOKEN', 'GITHUB_TOKEN'):
        try:
            GITHUB_TOKEN = userdata.get(secret_name)
        except Exception:
            GITHUB_TOKEN = None
        if GITHUB_TOKEN:
            break

git_environment = os.environ.copy()
if GITHUB_TOKEN:
    basic_credential = base64.b64encode(
        f'x-access-token:{GITHUB_TOKEN}'.encode('utf-8')
    ).decode('ascii')
    git_environment.update({
        'GIT_CONFIG_COUNT': '1',
        'GIT_CONFIG_KEY_0': 'http.extraHeader',
        'GIT_CONFIG_VALUE_0': f'Authorization: Basic {basic_credential}',
    })

def run_git(arguments):
    result = subprocess.run(
        ['git', *arguments], env=git_environment,
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        detail = result.stderr.strip() or result.stdout.strip() or 'erro desconhecido'
        hint = ''
        if not GITHUB_TOKEN:
            hint = '\nO repositório é privado: adicione GH_TOKEN aos Secrets do Colab.'
        raise RuntimeError(f'Git falhou ({result.returncode}): {detail}{hint}')
    return result

if IN_COLAB:
    if (PROJECT_ROOT / '.git').exists():
        dirty = run_git(
            ['-C', str(PROJECT_ROOT), 'status', '--porcelain']
        ).stdout.strip()
        if dirty:
            print('Clone possui artefatos persistentes; o pull continuará sem sobrescrevê-los.')
        run_git(['-C', str(PROJECT_ROOT), 'pull', '--ff-only', 'origin', 'main'])
    elif PROJECT_ROOT.exists() and any(PROJECT_ROOT.iterdir()):
        raise RuntimeError(f'O destino existe e não é um clone Git vazio: {PROJECT_ROOT}')
    else:
        run_git(['clone', '--depth', '1', '--branch', 'main', REPO_URL, str(PROJECT_ROOT)])

if not (PROJECT_ROOT / 'pyproject.toml').exists():
    raise FileNotFoundError('O clone não contém pyproject.toml; confirme o repositório e a branch main')

os.chdir(PROJECT_ROOT)

# Evita a mistura de arquivos Python e extensões binárias de versões diferentes do NumPy.
TARGET_NUMPY = '2.0.2'
numpy_was_loaded = 'numpy' in sys.modules
try:
    numpy_before = importlib.metadata.version('numpy')
except importlib.metadata.PackageNotFoundError:
    numpy_before = None
numpy_probe = subprocess.run(
    [sys.executable, '-c', 'import numpy; import numpy.char; print(numpy.__version__)'],
    capture_output=True, text=True,
)
repair_numpy = numpy_before != TARGET_NUMPY or numpy_probe.returncode != 0
if repair_numpy:
    print(f'Reparando NumPy: {numpy_before!r} -> {TARGET_NUMPY}')
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--force-reinstall',
        '--no-deps', f'numpy=={TARGET_NUMPY}',
    ], check=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-q',
    '-r', 'requirements.txt',
], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps',
], check=True)

dependency_probe = subprocess.run([
    sys.executable, '-c',
    'import numpy; import numpy.char; import transformers.generation.utils; '
    'print(numpy.__version__)',
], capture_output=True, text=True)
if dependency_probe.returncode != 0:
    raise RuntimeError('Ambiente incompatível após instalação:\n' + dependency_probe.stderr)
if repair_numpy and numpy_was_loaded:
    print('NumPy foi reparado. O runtime será reiniciado; depois execute o notebook novamente.')
    os.kill(os.getpid(), signal.SIGKILL)
print('Projeto:', PROJECT_ROOT)
print('Persistência:', DRIVE_ROOT)

In [ ]:
# Mantém datasets, checkpoints, adapters e avaliações no Google Drive.
persistent_directories = (
    'checkpoints/stage1', 'checkpoints/stage2', 'adapters', 'processed_data',
    'datasets/raw', 'datasets/interim', 'data_manifests', 'merged', 'logs', 'evaluations',
)
for directory in persistent_directories:
    (DRIVE_ROOT / directory).mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    links = {
        'data/raw': DRIVE_ROOT / 'datasets/raw',
        'data/interim': DRIVE_ROOT / 'datasets/interim',
        'data/processed': DRIVE_ROOT / 'processed_data',
        'data/manifests': DRIVE_ROOT / 'data_manifests',
        'outputs/checkpoints': DRIVE_ROOT / 'checkpoints',
        'outputs/adapters': DRIVE_ROOT / 'adapters',
        'outputs/merged': DRIVE_ROOT / 'merged',
        'outputs/logs': DRIVE_ROOT / 'logs',
        'outputs/evaluations': DRIVE_ROOT / 'evaluations',
    }
    for relative, target in links.items():
        local = PROJECT_ROOT / relative
        if local.is_symlink():
            continue
        existing = list(local.iterdir()) if local.exists() else []
        if any(item.name != '.gitkeep' for item in existing):
            raise RuntimeError(f'Diretório local não está vazio; dados preservados em: {local}')
        for item in existing:
            item.unlink()
        if local.exists():
            local.rmdir()
        local.parent.mkdir(parents=True, exist_ok=True)
        local.symlink_to(target, target_is_directory=True)

print('Diretórios persistentes preparados.')

## 2. Verificação da L4 e smoke test opcional do modelo

In [ ]:
import accelerate, datasets, peft, torch, transformers

if not torch.cuda.is_available():
    raise RuntimeError('Selecione Runtime > Change runtime type > GPU')
free, total = torch.cuda.mem_get_info()
environment = {
    'created_at': datetime.now(timezone.utc).isoformat(),
    'python': sys.version,
    'platform': platform.platform(),
    'gpu': torch.cuda.get_device_name(0),
    'vram_total_bytes': total,
    'vram_free_bytes': free,
    'bf16_supported': torch.cuda.is_bf16_supported(),
    'torch': torch.__version__,
    'cuda': torch.version.cuda,
    'transformers': transformers.__version__,
    'accelerate': accelerate.__version__,
    'peft': peft.__version__,
    'datasets': datasets.__version__,
}
print(json.dumps(environment, indent=2))
(DRIVE_ROOT / 'environment.json').write_text(
    json.dumps(environment, indent=2), encoding='utf-8'
)

if RUN_MODEL_SMOKE:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    name = 'Qwen/Qwen3-8B'
    tokenizer = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
    quant = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        name, quantization_config=quant, device_map={'': 0},
        attn_implementation='sdpa', trust_remote_code=True,
    )
    prompt = tokenizer.apply_chat_template(
        [{'role': 'user', 'content': 'Write a Python function that adds two integers.'}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    inputs = tokenizer(prompt, return_tensors='pt', add_special_tokens=False).to(model.device)
    with torch.inference_mode():
        output = model(**inputs)
    print('Forward logits:', tuple(output.logits.shape))
    del output, inputs, model
    torch.cuda.empty_cache()

## 3. Autenticação no Hugging Face, download e auditoria

Antes de executar, adicione `HF_TOKEN` em **Colab → Secrets** e habilite o acesso ao notebook.

In [ ]:
from huggingface_hub import login

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN and IN_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        HF_TOKEN = None
if not HF_TOKEN:
    raise RuntimeError('Adicione HF_TOKEN nos Secrets do Colab e habilite o acesso')
login(token=HF_TOKEN, add_to_git_credential=False)
os.environ['HF_TOKEN'] = HF_TOKEN
print('Hugging Face autenticado; o token não será exibido nem salvo nos manifests.')

In [ ]:
download_command = [
    sys.executable, 'scripts/download_datasets.py', '--config', 'configs/data.yaml'
]
if EXCLUDE_GLINT:
    download_command.append('--exclude-glint')
subprocess.run(download_command, check=True)
subprocess.run(
    [sys.executable, 'scripts/audit_datasets.py', '--config', 'configs/data.yaml'],
    check=True,
)

for manifest in sorted(Path('data/manifests').glob('*.json')):
    data = json.loads(manifest.read_text(encoding='utf-8'))
    revision = data.get('resolved_revision', data.get('splits'))
    license_name = data.get('license_card', data.get('license'))
    print(manifest.name, revision, license_name)

## 4. Preprocessing, deduplicação, splits e testes

In [ ]:
build_command = [
    sys.executable, 'scripts/build_examples.py', '--config', 'configs/data.yaml'
]
if EXCLUDE_GLINT:
    build_command.append('--exclude-glint')
if SMOKE_TEST:
    build_command += ['--limit', '750', '--max-output-examples', '750']
subprocess.run(build_command, check=True)
subprocess.run([
    sys.executable, 'scripts/deduplicate.py',
    '--input', 'data/interim/examples.jsonl',
    '--output', 'data/interim/examples.dedup.jsonl',
], check=True)
subprocess.run([
    sys.executable, 'scripts/split_by_group.py',
    '--input', 'data/interim/examples.dedup.jsonl',
    '--output-dir', 'data/processed',
], check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)
print(json.loads(Path('data/processed/split_manifest.json').read_text(encoding='utf-8')))

## 5. Stage 1 — QLoRA em 4096 tokens

Com `SMOKE_TEST=True`, executa no máximo 100 steps e 750 exemplos. Após validar a pipeline, altere para `False` e execute novamente para o treino principal.

In [ ]:
if RUN_STAGE1:
    free, total = torch.cuda.mem_get_info()
    print(torch.cuda.get_device_name(0), f'{free / 2**30:.2f}/{total / 2**30:.2f} GiB livres/total')
    train_command = [
        sys.executable, 'scripts/train_sft.py',
        '--config', 'configs/sft_stage1.yaml',
        '--resume-from-checkpoint', 'auto',
    ]
    if SMOKE_TEST:
        train_command += ['--max-steps', '100', '--max-train-samples', '750']
    subprocess.run(train_command, check=True)
else:
    print('Stage 1 ignorado por RUN_STAGE1=False')

## 6. Avaliação comparativa opcional

Ative `RUN_MODEL_COMPARISON` depois que o Stage 1 estiver concluído. A comparação usa os mesmos seeds e parâmetros para o modelo-base e os adapters existentes.

In [ ]:
if RUN_MODEL_COMPARISON:
    comparison_command = [
        sys.executable, 'scripts/evaluate_static.py',
        '--config', 'configs/eval.yaml',
        '--generate-models',
        '--output', 'outputs/evaluations/model_comparison.json',
    ]
    if SMOKE_TEST:
        comparison_command += ['--limit', '20']
    subprocess.run(comparison_command, check=True)
else:
    print('Comparação de modelos ignorada; defina RUN_MODEL_COMPARISON=True quando necessário.')

## 7. Stage 2 opcional — continuação em 8192 tokens

Só ative depois que o Stage 1 superar o modelo-base. A etapa exige `train_long.jsonl` e `validation_long.jsonl`, contendo apenas exemplos que realmente necessitam de contexto maior.

In [ ]:
if RUN_STAGE2:
    required_long_files = [
        Path('data/processed/train_long.jsonl'),
        Path('data/processed/validation_long.jsonl'),
    ]
    missing = [str(path) for path in required_long_files if not path.exists()]
    if missing:
        raise FileNotFoundError(
            'Stage 2 requer uma seleção auditada de exemplos longos: ' + ', '.join(missing)
        )
    subprocess.run([
        sys.executable, 'scripts/train_sft.py',
        '--config', 'configs/sft_stage2.yaml',
        '--resume-from-checkpoint', 'auto',
    ], check=True)
else:
    print('Stage 2 ignorado por RUN_STAGE2=False')

## 8. Exportação do adapter

In [ ]:
if RUN_EXPORT:
    stage2_adapter = Path('outputs/adapters/stage2')
    stage1_adapter = Path('outputs/adapters/stage1')
    adapter = stage2_adapter if stage2_adapter.exists() else stage1_adapter
    if not adapter.exists():
        raise FileNotFoundError('Nenhum adapter treinado foi encontrado para exportação')
    export_output = DRIVE_ROOT / 'adapters' / f'{adapter.name}-export'
    if export_output.exists():
        print('Export já existe:', export_output)
    else:
        subprocess.run([
            sys.executable, 'scripts/export_model.py',
            '--adapter', str(adapter),
            '--output', str(export_output),
        ], check=True)
else:
    print('Exportação ignorada; defina RUN_EXPORT=True para exportar o adapter aprovado.')

## 9. Agent harness opcional e resumo final

In [ ]:
agent_tasks = Path('data/processed/agent_tasks.jsonl')
if RUN_AGENT_EVAL:
    if not agent_tasks.exists():
        raise FileNotFoundError(f'Adicione tarefas executáveis em {agent_tasks}')
    subprocess.run([
        sys.executable, 'scripts/evaluate_agent.py',
        '--tasks', str(agent_tasks),
        '--output', 'outputs/evaluations/agent_runs.jsonl',
    ], check=True)

summary = {
    'project': str(PROJECT_ROOT),
    'persistent_root': str(DRIVE_ROOT),
    'stage1_adapter': Path('outputs/adapters/stage1').exists(),
    'stage2_adapter': Path('outputs/adapters/stage2').exists(),
    'comparison': Path('outputs/evaluations/model_comparison.json').exists(),
}
print(json.dumps(summary, indent=2, ensure_ascii=False))